# ODYSSEY → CIRRUS level correction (H3)

H3 (#20): ODYSSEY and CIRRUS differ in level, and only 4 days overlap. The ticket compares three variants with
the same criteria: a pan-European bridge, a national bridge, and plain seasonal quantile mapping between the two
products (`docs/method.md`, *Level correction (H3)*). The bridge data are not on this PC yet (S7, #77), so this
notebook starts with what ODYSSEY and CIRRUS alone can show.

1. **Distributions over Belgium**, per month and hour of day, for ODYSSEY (2 km / 15 min) and CIRRUS (1 km /
   5 min, and degraded to 2 km).
2. **The overlap days**, 1–5/7/2024, on paired times: the size and shape of the difference.
3. **Calendar months across years:** can two years of CIRRUS give a seasonal quantile mapping?
4. **Hourly radar rain on the ERA5-Land grid,** ready for the pan-European bridge.
5. **Bridge mapping, tested on synthetic data.**
6. **Plain seasonal quantile mapping on the overlap days.**

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import datetime as dt
import glob
import os

import matplotlib.pyplot as plt
import numpy as np

from opera_ds.odim_io import read_odim_composite

OUT = RESULTS_DIR / "h3_level"
OUT.mkdir(parents=True, exist_ok=True)
FIG = DATA_DIR.parent / "docs" / "figures" / "h3-level"
FIG.mkdir(parents=True, exist_ok=True)

## 1. Distributions over Belgium

- **Belgium:** the Natural Earth mask `data/mask_BEL_1km.npy` (30,667 pixels at 1 km). At 2 km a cell is
  Belgian when at least 2 of its 4 pixels are (7,744 cells).
- **Data:** the crops of `docs/data.md`, *Derived datasets*: ODYSSEY `DBZH_QIND` 9/2012–7/2024 and CIRRUS `DBZH`
  7/2024–11/2026. CIRRUS is also degraded to 2 km by `zmean`, the degradation closest to real ODYSSEY (H1,
  `docs/method.md`, *Degraded CIRRUS as a stand-in for ODYSSEY*).
- **Counts per frame**, inside the coverage only: `dry` (undetect), `gt70` (above 70 dBZ, impossible values, H2)
  and a histogram of the other echoes from −10 to 70 dBZ in 0.5 dB bins. No other cleaning: #20 starts with the
  cap only.
- **Kept per month,** per hour of day and per kind of time: index 1 for minutes 00/15/30/45 (the ODYSSEY times),
  0 for the other CIRRUS times.
- **Cost:** about 15 ms per ODYSSEY frame and 20 ms per CIRRUS frame. In one process, all 640,000 frames take
  about 3 h; with 16 processes (script version, 11/11/2026) about 15 min. Months already written are skipped.

In [ ]:
EDGES = np.arange(-10.0, 70.001, 0.5)
CENT = (EDGES[:-1] + EDGES[1:]) / 2
NB = len(EDGES) - 1
MASK = np.load(DATA_DIR / "mask_BEL_1km.npy")
_r, _c = np.where(MASK.any(1))[0], np.where(MASK.any(0))[0]
R0, R1 = _r[0] // 2 * 2, (_r[-1] + 2) // 2 * 2      # even edges: the window maps onto the 2 km grid
C0, C1 = _c[0] // 2 * 2, (_c[-1] + 2) // 2 * 2
M1 = MASK[R0:R1, C0:C1]
M2 = M1.reshape(M1.shape[0] // 2, 2, M1.shape[1] // 2, 2).sum((1, 3)) >= 2
print("Belgian pixels at 1 km:", M1.sum(), " cells at 2 km:", M2.sum())


def empty():
    z = lambda: np.zeros((24, 2), np.int64)  # noqa: E731
    return {"n": z(), "valid": z(), "dry": z(), "gt70": z(), "hist": np.zeros((24, 2, NB), np.int64)}


def add(acc, h, q, z, cov, und, m):
    ok = cov & m
    det = ok & ~und & np.isfinite(z)
    acc["n"][h, q] += 1
    acc["valid"][h, q] += ok.sum()
    acc["dry"][h, q] += (ok & und).sum()
    acc["gt70"][h, q] += (det & (z > 70)).sum()
    acc["hist"][h, q] += np.histogram(np.clip(z[det & (z <= 70)], EDGES[0], EDGES[-1]), EDGES)[0]


def window(f, r0, r1, c0, c1):
    a, b = int(f.where.get("crop_row0", 0)), int(f.where.get("crop_col0", 0))
    s = (slice(r0 - a, r1 - a), slice(c0 - b, c1 - b))
    return f.values[s].astype(np.float64), f.coverage[s], f.undetect[s]


def zmean2(z, cov, und):
    """2 x 2 mean in linear Z, as opera_ds.validate.degrade_frame(how='zmean')."""
    sh = (z.shape[0] // 2, 2, z.shape[1] // 2, 2)
    det = cov & ~und
    c = cov.reshape(sh).all((1, 3))
    u = c & ~det.reshape(sh).any((1, 3))
    lin = np.where(det, np.power(10.0, np.nan_to_num(z) / 10.0), 0.0).reshape(sh).mean((1, 3))
    out = np.zeros(c.shape)
    out[c & ~u] = 10 * np.log10(lin[c & ~u])
    return out, c, u


def month_hist(product, ym):
    path = OUT / f"hist_{product}_{ym}.npz"
    if path.exists():
        return
    y, m = ym.split("-")
    files = sorted(glob.glob(str(DATA_DIR / f"{product}_bel" / y / m / "*.h5")))
    accs = {"native": empty()} if product == "odyssey" else {"native": empty(), "deg2": empty()}
    bad = 0
    for p in files:
        try:
            f = read_odim_composite(p, "DBZH")
        except Exception:  # noqa: BLE001
            bad += 1
            continue
        h, q = f.time.hour, int(f.time.minute % 15 == 0)
        if product == "odyssey":
            add(accs["native"], h, q, *window(f, R0 // 2, R1 // 2, C0 // 2, C1 // 2), M2)
        else:
            z, cov, und = window(f, R0, R1, C0, C1)
            add(accs["native"], h, q, z, cov, und, M1)
            add(accs["deg2"], h, q, *zmean2(z, cov, und), M2)
    flat = {f"{k}_{n}": v for k, a in accs.items() for n, v in a.items()}
    np.savez_compressed(path, edges=EDGES, n_files=len(files), n_bad=bad, **flat)


def months_on_disk(product):
    root = DATA_DIR / f"{product}_bel"
    return sorted(f"{os.path.basename(os.path.dirname(d))}-{os.path.basename(d)}"
                  for d in glob.glob(str(root / "*" / "*")) if os.path.isdir(d))


for product in ("odyssey", "cirrus"):
    for ym in months_on_disk(product):
        month_hist(product, ym)
print(len(glob.glob(str(OUT / "hist_odyssey_20*.npz"))), "ODYSSEY months,",
      len(glob.glob(str(OUT / "hist_cirrus_20*.npz"))), "CIRRUS months")

In [ ]:
def load(product, key="native", quarter=None):
    """{YYYY-MM: (histogram, valid pixels)} summed over hours; `quarter` 1 keeps only the ODYSSEY times."""
    out = {}
    for f in sorted(glob.glob(str(OUT / f"hist_{product}_20*.npz"))):
        z = np.load(f)
        h, v = z[f"{key}_hist"], z[f"{key}_valid"]
        h, v = (h[:, quarter], v[:, quarter]) if quarter is not None else (h.sum(1), v.sum(1))
        out[os.path.basename(f)[-11:-4]] = (h.sum(0), v.sum())
    return out


def wet(h, v, thr=10.0):
    """Share of the valid pixels with an echo of at least `thr` dBZ."""
    return h[CENT >= thr].sum() / max(v, 1)


def quant(h, p):
    """Quantile `p` of the echoes (bin centre)."""
    c = np.cumsum(h) / max(h.sum(), 1)
    return CENT[min(np.searchsorted(c, p), NB - 1)]


ODY = load("odyssey")
CIR = load("cirrus", "deg2", quarter=1)       # 2 km, ODYSSEY times
CIR1 = load("cirrus", "native")               # 1 km, all times

## 2. The overlap days, 1–5/7/2024

ODYSSEY at NT is paired with CIRRUS at NT+5, the closest window (`docs/data.md`, *Time windows*). Only cells
covered by both products count. Statistics are on all echoes, as in `docs/method.md`, *Degraded CIRRUS as a
stand-in for ODYSSEY*, and on echoes of at least 10 dBZ.

In [ ]:
cir_files = {os.path.basename(p).split("@")[1]: p for p in glob.glob(str(DATA_DIR / "cirrus_bel" / "2024" / "07" / "*.h5"))}
pair_o, pair_c = empty(), empty()
n_pairs = 0
for p in sorted(glob.glob(str(DATA_DIR / "odyssey_bel" / "2024" / "07" / "*.h5"))):
    t = dt.datetime.strptime(os.path.basename(p).split("@")[1], "%Y%m%dT%H%M")
    k = (t + dt.timedelta(minutes=5)).strftime("%Y%m%dT%H%M")
    if k not in cir_files:
        continue
    zo = window(read_odim_composite(p, "DBZH"), R0 // 2, R1 // 2, C0 // 2, C1 // 2)
    zd = zmean2(*window(read_odim_composite(cir_files[k], "DBZH"), R0, R1, C0, C1))
    both = M2 & zo[1] & zd[1]
    add(pair_o, 0, 0, *zo, both)
    add(pair_c, 0, 0, *zd, both)
    n_pairs += 1
ho, vo = pair_o["hist"][0, 0], pair_o["valid"][0, 0]
hc, vc = pair_c["hist"][0, 0], pair_c["valid"][0, 0]
print(n_pairs, "pairs")
print("share of pixels with echoes >= 10 dBZ: ODYSSEY", round(wet(ho, vo), 3), " CIRRUS", round(wet(hc, vc), 3))
P = [0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
for name, sel in (("all echoes", CENT > -99), ("echoes >= 10 dBZ", CENT >= 10)):
    qo = [quant(np.where(sel, ho, 0), p) for p in P]
    qc = [quant(np.where(sel, hc, 0), p) for p in P]
    print(f"{name:17s} quantiles {P}")
    print("   ODYSSEY       ", qo)
    print("   CIRRUS 2 km   ", qc)
    print("   CIRRUS−ODYSSEY", [round(b - a, 1) for a, b in zip(qo, qc)])

## 3. Calendar months across years

A plain seasonal quantile mapping would fit, for each month of the year, the ODYSSEY distribution onto the CIRRUS
one. ODYSSEY is taken from its last period (8/2018–6/2024, `docs/method.md`, *Radars over time*) and CIRRUS from
8/2024–11/2026 at 2 km on the ODYSSEY times; July 2024 is left out of both, since it holds the overlap days. The
question is whether the product difference stands out of the year-to-year spread of each month.

In [ ]:
def monthly(series, lo, hi, skip=()):
    out = {}
    for ym, (h, v) in series.items():
        if lo <= ym <= hi and ym not in skip:
            out.setdefault(int(ym[5:]), []).append((ym, h, v))
    return out


mo = monthly(ODY, "2018-08", "2024-06")
mc = monthly(CIR, "2024-08", "2026-11")
print("month  ODYSSEY: years, share >= 10 dBZ, median (min-max of the years) | CIRRUS: years, share, median per year")
for m in range(1, 13):
    yo = [(wet(h, v), quant(h, 0.5)) for _, h, v in mo[m]]
    yc = [(wet(h, v), quant(h, 0.5)) for _, h, v in mc[m]]
    so, sc = [a for a, _ in yo], [a for a, _ in yc]
    qo, qc = [b for _, b in yo], [b for _, b in yc]
    print(f"{m:2d}   {len(yo)}  {min(so):.3f}-{max(so):.3f}  {min(qo):5.2f}-{max(qo):5.2f} dBZ | "
          f"{len(yc)}  {', '.join(f'{s:.3f}' for s in sc)}  {', '.join(f'{q:5.2f}' for q in qc)} dBZ")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), constrained_layout=True)
for ax, (stat, label) in zip(axes, ((lambda h, v: wet(h, v), "share of Belgian pixels with echoes ≥ 10 dBZ"),
                                    (lambda h, v: quant(h, 0.5), "median of the echoes (dBZ)"))):
    for m in range(1, 13):
        so_ = [stat(h, v) for _, h, v in mo[m]]
        sc_ = [stat(h, v) for _, h, v in mc[m]]
        ax.plot([m - 0.12] * len(so_), so_, "o", ms=4, color="tab:blue", alpha=0.6,
                label="ODYSSEY, one dot per year 8/2018–6/2024" if m == 1 else None)
        ax.plot([m + 0.12] * len(sc_), sc_, "s", ms=5, color="tab:red",
                label="CIRRUS 2 km, one dot per year 8/2024–11/2026" if m == 1 else None)
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("month")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
o_d = [wet(ho, vo), quant(ho, 0.5)]
c_d = [wet(hc, vc), quant(hc, 0.5)]
for ax, a, b in zip(axes, o_d, c_d):
    ax.plot(7, a, "*", ms=13, color="tab:blue", mec="k", label="ODYSSEY, 1–5/7/2024 (paired times)")
    ax.plot(7, b, "*", ms=13, color="tab:red", mec="k", label="CIRRUS, 1–5/7/2024 (paired times)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=2, fontsize=9)
fig.suptitle("ODYSSEY and CIRRUS over Belgium: the year-to-year spread of a month is wider than the product gap")
fig.savefig(FIG / "months-odyssey-cirrus.png", dpi=120)
plt.show()

## 4. Hourly radar rain on the ERA5-Land grid

Ready for the pan-European bridge (S7, #77): the ERA5-Land grid of `scripts/fetch_cds.py`, cell centres at
55.0 … 45.6 N and 3.9 W … 11.8 E, step 0.1° (95 × 158).

- A radar pixel belongs to the cell that contains its centre; only Belgian pixels count.
- **Rain rate:** Marshall–Palmer (Z = 200 R^1.6) from 7 dBZ, capped at 55 dBZ (Imhoff et al. 2020); undetect
  and values above 70 dBZ give no rain.
- **Hours:** each frame stands for the step that ends at its label (5 min CIRRUS, 15 min ODYSSEY), so the hour
  ending at H sums the labels H−55 … H (12 CIRRUS frames) or H−45 … H (4 ODYSSEY frames), as ERA5-Land hourly
  amounts end at their time stamp. The 2.5 min offset of the ODYSSEY window (`docs/data.md`, *Time windows*) is
  ignored. Hours with missing frames are kept with their frame count, to be left out later.
- **Per cell and hour:** the mean rain (mm) of its covered Belgian pixels. Cells with at least 80% of the largest
  pixel count are called full (186 at 2 km, 361 at 1 km).
- **Cost:** about 10 ms per frame (about 2 h in one process; 10 min with 16 processes, 11/11/2026).
- **Caveat:** the composite is a column maximum with no cleaning, so these amounts are an index. In July 2023
  the Belgian mean is about 370 mm, an index of the column maximum rather than rain at the ground; the bridge
  comparison therefore matches distributions, not amounts.

In [ ]:
LAT_N, LON_W, STEP, NLAT, NLON = 55.0, -3.9, 0.1, 95, 158
STEP_MIN = {"odyssey": 15, "cirrus": 5}
FULL_FRAMES = {"odyssey": 4, "cirrus": 12}


def cell_index(where, r0, r1, c0, c1, m):
    """ERA5-Land cell (flat index) of each pixel centre of the window [r0:r1, c0:c1]; -1 outside or not Belgian."""
    from pyproj import Proj

    p = Proj(where["projdef"])
    sx, sy = float(where["xscale"]), float(where["yscale"])
    xu, yu = p(float(where["UL_lon"]), float(where["UL_lat"]))
    a, b = int(where.get("crop_row0", 0)), int(where.get("crop_col0", 0))
    rr, cc = np.mgrid[r0 - a:r1 - a, c0 - b:c1 - b]
    lon, lat = p(xu + (cc + 0.5) * sx, yu - (rr + 0.5) * sy, inverse=True)
    i = np.rint((LAT_N - lat) / STEP).astype(int)
    j = np.rint((lon - LON_W) / STEP).astype(int)
    ok = m & (i >= 0) & (i < NLAT) & (j >= 0) & (j < NLON)
    return np.where(ok, i * NLON + j, -1)


def rain_rate(z, cov, und):
    zz = np.nan_to_num(z, nan=0.0)
    r = np.where((zz >= 7) & (zz <= 70) & ~und, (10 ** (np.minimum(zz, 55.0) / 10) / 200.0) ** (1 / 1.6), 0.0)
    return np.where(cov, r, np.nan)


def month_era5grid(product, ym):
    path = OUT / f"era5grid_{product}_{ym}.npz"
    if path.exists():
        return
    import calendar

    y, m = map(int, ym.split("-"))
    files = sorted(glob.glob(str(DATA_DIR / f"{product}_bel" / f"{y}" / f"{m:02d}" / "*.h5")))
    nh = calendar.monthrange(y, m)[1] * 24
    start = dt.datetime(y, m, 1, 1)                     # the first hour ends at 01:00
    win = (R0 // 2, R1 // 2, C0 // 2, C1 // 2) if product == "odyssey" else (R0, R1, C0, C1)
    mask = M2 if product == "odyssey" else M1
    idx = None
    frames = np.zeros(nh, np.int32)
    step_h = STEP_MIN[product] / 60.0
    for p in files:
        try:
            f = read_odim_composite(p, "DBZH")
        except Exception:  # noqa: BLE001
            continue
        if idx is None:
            idx = cell_index(f.where, *win, mask)
            sel = idx >= 0
            cells, inv = np.unique(idx[sel], return_inverse=True)
            npix = np.bincount(inv, minlength=len(cells))
            rain_h = np.zeros((nh, len(cells)))
            cover_h = np.zeros((nh, len(cells)), np.int64)
        k = int(np.ceil((f.time - start).total_seconds() / 3600.0))
        if not 0 <= k < nh:
            continue
        a, b = int(f.where.get("crop_row0", 0)), int(f.where.get("crop_col0", 0))
        s = (slice(win[0] - a, win[1] - a), slice(win[2] - b, win[3] - b))
        r = rain_rate(f.values[s].astype(np.float64), f.coverage[s], f.undetect[s])[sel]
        ok = np.isfinite(r)
        rain_h[k] += np.bincount(inv[ok], weights=r[ok] * step_h, minlength=len(cells))
        cover_h[k] += np.bincount(inv[ok], minlength=len(cells))
        frames[k] += 1
    if idx is None:
        return
    mean_rain = rain_h * frames[:, None] / np.maximum(cover_h, 1)
    hours = [(start + dt.timedelta(hours=k)).isoformat() for k in range(nh)]
    np.savez_compressed(path, rain=mean_rain.astype(np.float32), frames=frames, cover=cover_h, npix=npix,
                        cells=np.stack([cells // NLON, cells % NLON], 1), hour_end=np.array(hours))


for product in ("odyssey", "cirrus"):
    for ym in months_on_disk(product):
        month_era5grid(product, ym)

In [ ]:
series = {}
for product in ("odyssey", "cirrus"):
    for f in sorted(glob.glob(str(OUT / f"era5grid_{product}_20*.npz"))):
        z = np.load(f)
        ok = z["frames"] == FULL_FRAMES[product]
        if ok.mean() < 0.5:                        # e.g. ODYSSEY 7/2024 (5 days), CIRRUS 10/2024 (gap)
            continue
        full = z["npix"] >= 0.8 * z["npix"].max()
        r = z["rain"][ok][:, full]
        ym = os.path.basename(f)[-11:-4]
        series[(product, ym)] = (r.mean() * 24 * 30.44, (r >= 0.1).mean())

fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True, constrained_layout=True)
for product, colour in (("odyssey", "tab:blue"), ("cirrus", "tab:red")):
    keys = sorted(k for k in series if k[0] == product)
    t = [dt.datetime.strptime(k[1], "%Y-%m") for k in keys]
    axes[0].plot(t, [series[k][0] for k in keys], ".-", color=colour, lw=0.8, label=product.upper())
    axes[1].plot(t, [series[k][1] for k in keys], ".-", color=colour, lw=0.8)
for ax in axes:
    for s in (dt.datetime(2015, 12, 1), dt.datetime(2018, 8, 1), dt.datetime(2024, 7, 1)):
        ax.axvline(s, color="0.4", ls="--", lw=0.8)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("radar rain index\n(mm per 30.44 days)")
axes[1].set_ylabel("share of cell-hours\nwith ≥ 0.1 mm")
axes[0].legend(loc="upper left", fontsize=9)
fig.suptitle("Belgium on the ERA5-Land grid, complete hours only; dashed lines: OPERA changes of 12/2015, 8/2018 and 7/2024")
fig.savefig(FIG / "monthly-radar-era5grid.png", dpi=120)
plt.show()

## 5. Bridge mapping, tested on synthetic data

How a bridge removes the product gap even when the weather differs between the two periods.

1. **ODYSSEY → bridge:** each ODYSSEY value goes to the bridge value with the same cumulative probability,
   both taken over the ODYSSEY years.
2. **Bridge → CIRRUS:** that bridge value goes to the CIRRUS value with the same cumulative probability, both
   taken over the CIRRUS years.
3. **Dry stays dry:** a dry ODYSSEY hour stays dry. The wet frequency is matched through the threshold pair
   (Rosenfeld et al. 1993, section 5a), not by adding rain at random as in the frequency adaptation of Themeßl
   et al. (2012): the correction must stay deterministic.

The same weather series feeds the bridge and a product in each period. ODYSSEY reads half as strong as CIRRUS
and misses weak rain; the truth is what CIRRUS would have seen in the ODYSSEY years. Scores are taken above
the level ODYSSEY still detects, since rain it never sees cannot be restored.

In [ ]:
def prob_map(x, src, dst):
    """Value of `dst` with the same cumulative probability that `x` has in `src` (empirical quantile mapping)."""
    s = np.sort(src)
    p = np.searchsorted(s, x, side="right") / len(s)
    return np.quantile(np.sort(dst), np.clip(p, 0, 1))


def bridge_map(x, src, bridge_src, dst, bridge_dst):
    """src -> bridge level (src years), bridge level -> dst (dst years); dry stays dry."""
    level = prob_map(x, src, bridge_src)
    level[x <= 0] = 0.0
    out = prob_map(level, bridge_dst, dst)
    out[level <= 0] = 0.0
    return out


rng = np.random.default_rng(1)


def weather(n, wet_share, scale):
    return np.where(rng.random(n) < wet_share, rng.gamma(0.7, scale, n), 0.0)


def product(b, gain, floor):
    r = b * gain
    r[r < floor] = 0.0
    return r


n, level = 400_000, 1.2                      # level: what ODYSSEY still detects, in CIRRUS units
print("weather of the CIRRUS years | share >= level: truth, plain, bridge | quantiles 50/90/99 over truth: plain, bridge")
for wet_share, scale in ((0.20, 1.5), (0.12, 1.0), (0.28, 2.0)):
    b_ody, b_cir = weather(n, 0.20, 1.5), weather(n, wet_share, scale)
    ody, cir, truth = product(b_ody, 1.0, 0.6), product(b_cir, 2.0, 0.1), product(b_ody, 2.0, 0.1)
    plain = prob_map(ody, ody, cir)
    plain[ody <= 0] = 0.0
    bridged = bridge_map(ody, ody, b_ody, cir, b_cir)
    share = lambda a: (a >= level).mean()                                   # noqa: E731
    q = lambda a: np.quantile(a[a >= level], [0.5, 0.9, 0.99])               # noqa: E731
    print(f"wet {wet_share:.2f}, scale {scale}: {share(truth):.3f} {share(plain):.3f} {share(bridged):.3f} | "
          f"{np.round(q(plain) / q(truth), 2)} {np.round(q(bridged) / q(truth), 2)}")

## 6. Plain seasonal quantile mapping on the overlap days

The third variant, scored with the first test of `docs/method.md`, *Evaluation criteria*. The ODYSSEY
distribution of the fitting months (period 8/2018–6/2024, all pixels, dry included) is mapped onto the CIRRUS
distribution of the same calendar months (2 km, ODYSSEY times). Dry stays dry. July 2024, which holds the
overlap days, is left out of both. The mapping is then applied to ODYSSEY on the 382 paired times of section 2.

In [ ]:
VALS = np.concatenate([[-99.0], CENT])            # index 0: dry


def dist(series_key, product, key, quarter, months):
    h = np.zeros(NB + 1)
    for ym in months:
        z = np.load(OUT / f"hist_{product}_{ym}.npz")
        hh, dd = z[f"{key}_hist"], z[f"{key}_dry"]
        hh, dd = (hh[:, quarter], dd[:, quarter]) if quarter is not None else (hh.sum(1), dd.sum(1))
        h += np.concatenate([[dd.sum()], hh.sum(0)])
    return h


def plain_mapping(src, dst):
    """For each source bin (dry first), the target value at the same mid-bin cumulative probability."""
    cs, cd = np.cumsum(src) / src.sum(), np.cumsum(dst) / dst.sum()
    mid = cs - src / src.sum() / 2
    out = VALS[np.minimum(np.searchsorted(cd, mid), len(VALS) - 1)]
    out[0] = -99.0                                 # dry stays dry
    return out


def scores(values, weights):
    tot = weights.sum()
    share = [weights[values >= t].sum() / tot for t in (10, 20, 30)]
    det = values > -99
    o = np.argsort(values[det])
    v, w = values[det][o], weights[det][o]
    c = np.cumsum(w) / w.sum()
    return share, [v[np.searchsorted(c, p)] for p in (0.25, 0.5, 0.75, 0.9, 0.99)]


ov_o = np.concatenate([[pair_o["dry"][0, 0]], ho])
ov_c = np.concatenate([[pair_c["dry"][0, 0]], hc])
ref = scores(VALS, ov_c)
rows = [("CIRRUS (target)", ref), ("ODYSSEY, uncorrected", scores(VALS, ov_o))]
ody_july = [f"{y}-07" for y in range(2019, 2024)]
fits = {
    "July, CIRRUS 2025 and 2026": (ody_july, ["2025-07", "2026-07"]),
    "July, CIRRUS 2025 only": (ody_july, ["2025-07"]),
    "June-August, all CIRRUS": ([f"{y}-{m:02d}" for y in range(2019, 2024) for m in (6, 7, 8)],
                                ["2024-08", "2025-06", "2025-07", "2025-08", "2026-06", "2026-07", "2026-08"]),
}
for name, (om, cm) in fits.items():
    mp = plain_mapping(dist(None, "odyssey", "native", None, om), dist(None, "cirrus", "deg2", 1, cm))
    rows.append((f"plain mapping, {name}", scores(mp, ov_o)))
print(f"{'':42s} share >= 10/20/30 dBZ      CIRRUS minus row at q25/50/75/90/99 (dB)")
for name, (share, q) in rows:
    print(f"{name:42s} {share[0]:.3f} {share[1]:.3f} {share[2]:.3f}    "
          + " ".join(f"{a - b:+5.1f}" for a, b in zip(ref[1], q)))

## Results so far (11/11/2026)

- **Overlap days.** On the 382 paired times, CIRRUS has echoes of at least 10 dBZ on 22.6% of the Belgian
  pixels, ODYSSEY on 15.6%. Over all echoes, CIRRUS is higher by 1–5 dB depending on the quantile (median +4 dB,
  99th percentile +5 dB). Over echoes of at least 10 dBZ the shift is only 0.5–1.5 dB up to the 90th
  percentile: most of the gap is in **how often** ODYSSEY sees rain, not in how strong it sees it.
- **Calendar months.** Across years, the same month changes much more than the products differ: in June the
  ODYSSEY share of pixels with echoes ≥ 10 dBZ goes from 7.5% (2023) to 25% (2020), and the median from 7 to
  15 dBZ. The CIRRUS months fall inside the ODYSSEY spread, often at its low end (dry spring 2025).
- **Consequence for the third variant.** With two years of CIRRUS, a seasonal quantile mapping between the two
  products would copy the weather of 2024–2026 into the ODYSSEY years instead of removing the product
  difference. This is the reason for a bridge in #20; the plain mapping stays as the reference the bridges must
  beat. On the overlap days it moves ODYSSEY away from CIRRUS when fitted on all CIRRUS Julys (July 2026 is dry),
  and closer when fitted on July 2025 alone: the result depends on the CIRRUS year (section 6).
- **July 2026.** CIRRUS has echoes on only 5% of the Belgian pixel-times (95% undetect), against 17–26% in the
  other summer months; the Belgian radars are listed in `how/nodes`. Not checked against gauges: a dry month or
  a data problem.